[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/prefect-certified/notebooks/day-05-subflows-modular.ipynb#scrollTo=a1f2b3c4)

---
# Day 5 · Subflows and Modular Pipeline Design
**certified-journeys / prefect-certified** · Learn · Prefect for Data Engineers

> **Goal for today:** By the end of this notebook you can decompose a monolithic pipeline into a parent flow that orchestrates focused `@flow`-decorated subflows, pass parameters between them, inspect subflow states programmatically with `return_state=True`, and understand how subflow failures propagate to the parent.

In [ ]:
%pip install -q "prefect>=2.14"

## Step 1 · What is a subflow?

Any `@flow`-decorated function called **from inside another flow** becomes a **subflow**. Prefect tracks it as a separate flow run nested under the parent run.

```python
@flow
def ingest() -> list:
    ...

@flow
def pipeline():
    data = ingest()   # ← subflow call
```

Key properties:

| Property | Detail |
|---|---|
| Independent run record | Each subflow gets its own flow run ID, state, and logs |
| Parameters | Passed as normal function arguments |
| Return value | Returned like any Python function call |
| State propagation | A failed subflow raises in the parent by default |
| UI visibility | Shown as nested runs under the parent in the Runs view |

> **Why subflows, not just tasks?** Tasks are the right unit for a single data operation. Subflows are the right unit for a *group of related tasks* that form a logical stage — think Ingest, Transform, Load.

In [ ]:
from prefect import flow, task


# ── Minimal subflow example ───────────────────────────────────────────────────

@task
def extract_rows(source: str) -> list[dict]:
    """Simulates reading rows from a data source."""
    print(f"  Extracting from {source!r}")
    return [{"id": i, "value": i * 10, "source": source} for i in range(1, 4)]


@flow(name="ingest-subflow", log_prints=True)
def ingest_flow(source: str) -> list[dict]:
    """Subflow: pulls raw rows from a source."""
    rows = extract_rows(source)
    print(f"  ingest_flow fetched {len(rows)} rows")
    return rows


@flow(name="parent-pipeline", log_prints=True)
def parent_pipeline():
    """Parent flow that calls ingest_flow as a subflow."""
    print("Parent: starting")
    rows = ingest_flow(source="orders_2024")   # subflow call
    print(f"Parent: received {len(rows)} rows from subflow")
    return rows


result = parent_pipeline()
print(f"\nFinal result from parent: {result}")

### What just happened?

- `ingest_flow` ran as a **nested flow run** inside `parent_pipeline` — both received their own run IDs.
- The parent called the subflow just like any Python function and got its return value back.
- **Both flows are `@flow`-decorated**, so both produce observable run records with full state history.
- In the Prefect UI the parent run shows a "Subflow runs" section listing `ingest-subflow`.

## Step 2 · Refactoring a monolith into domain subflows

A large pipeline is easier to maintain when split by **domain boundary**: Ingest → Transform → Load. Each subflow can be:

- **Scheduled and run independently** for partial re-runs
- **Unit-tested** in isolation
- **Retried** at the subflow level without re-running upstream stages

Pattern:

```
parent_pipeline()
  └─ ingest_flow(source)          → raw rows
  └─ transform_flow(rows, config) → cleaned rows
  └─ load_flow(rows, target)      → record count loaded
```

In [ ]:
from prefect import flow, task


# ── Ingest subflow ────────────────────────────────────────────────────────────

@task
def read_csv_task(path: str) -> list[dict]:
    """Simulates reading a CSV; returns raw dicts."""
    return [
        {"id": 1, "amount": "12.5",  "status": "ok"},
        {"id": 2, "amount": "bad",   "status": "ok"},
        {"id": 3, "amount": "8.0",   "status": "skip"},
        {"id": 4, "amount": "99.9",  "status": "ok"},
    ]


@flow(name="ingest", log_prints=True)
def ingest_flow(path: str) -> list[dict]:
    rows = read_csv_task(path)
    print(f"  [ingest] read {len(rows)} raw rows from {path!r}")
    return rows


# ── Transform subflow ─────────────────────────────────────────────────────────

@task
def clean_task(rows: list[dict], drop_status: str) -> list[dict]:
    """Drops rows by status and coerces amount to float."""
    cleaned = []
    for r in rows:
        if r["status"] == drop_status:
            continue
        try:
            cleaned.append({**r, "amount": float(r["amount"])})
        except ValueError:
            print(f"    [transform] skipping non-numeric amount in row {r['id']}")
    return cleaned


@flow(name="transform", log_prints=True)
def transform_flow(raw_rows: list[dict], drop_status: str = "skip") -> list[dict]:
    cleaned = clean_task(raw_rows, drop_status)
    print(f"  [transform] {len(cleaned)} rows after cleaning (drop_status={drop_status!r})")
    return cleaned


# ── Load subflow ──────────────────────────────────────────────────────────────

@task
def write_task(rows: list[dict], target: str) -> int:
    """Simulates writing rows to a destination; returns the row count."""
    print(f"    [load] writing {len(rows)} rows to {target!r}")
    return len(rows)


@flow(name="load", log_prints=True)
def load_flow(rows: list[dict], target: str) -> int:
    count = write_task(rows, target)
    print(f"  [load] wrote {count} rows")
    return count


# ── Parent pipeline ───────────────────────────────────────────────────────────

@flow(name="etl-pipeline", log_prints=True)
def etl_pipeline(source_path: str, target: str, drop_status: str = "skip") -> dict:
    # Parameters flow from parent to each subflow via regular arguments
    raw      = ingest_flow(path=source_path)
    cleaned  = transform_flow(raw_rows=raw, drop_status=drop_status)
    loaded   = load_flow(rows=cleaned, target=target)
    summary  = {"source": source_path, "target": target, "rows_loaded": loaded}
    print(f"Pipeline summary: {summary}")
    return summary


etl_pipeline(source_path="/data/orders.csv", target="warehouse.orders")

### What just happened?

- The monolith was split into three focused subflows: **ingest → transform → load**.
- **Parameters** (`source_path`, `drop_status`, `target`) were passed from the parent to each subflow as ordinary function arguments — no special API required.
- The parent's return value is assembled from the subflows' return values.
- **Each subflow is independently deployable**: you could schedule `ingest` on its own without running the full pipeline.

## Step 3 · Passing parameters between parent and subflows

Subflows receive parameters the same way any Python function does. The important nuances:

| Scenario | How |
|---|---|
| Pass a scalar (str, int, bool) | Direct argument — no serialisation needed |
| Pass a list / dict from a previous subflow | Return from subflow A, pass to subflow B |
| Share a config object | Build it in the parent, pass to each subflow |
| Avoid large payloads | Keep large DataFrames inside tasks; pass references between flows |

> Prefect serialises return values when they cross a flow boundary (for state storage). For very large objects, consider writing to a file system and passing the path.

In [ ]:
from dataclasses import dataclass
from prefect import flow, task


@dataclass
class PipelineConfig:
    """Shared config object passed from parent to every subflow."""
    env: str           # "dev" | "prod"
    batch_size: int
    dry_run: bool


@task
def fetch_batch(config: PipelineConfig, offset: int) -> list[int]:
    """Returns a batch of IDs; respects batch_size from config."""
    ids = list(range(offset, offset + config.batch_size))
    print(f"    fetch_batch offset={offset} → {ids}")
    return ids


@flow(name="fetch-subflow", log_prints=True)
def fetch_subflow(config: PipelineConfig, offset: int) -> list[int]:
    return fetch_batch(config, offset)


@task
def process_ids(ids: list[int], config: PipelineConfig) -> list[str]:
    """Converts IDs to record strings; skips actual write if dry_run."""
    records = [f"{config.env}:{i}" for i in ids]
    if config.dry_run:
        print(f"    [dry-run] would write: {records}")
    return records


@flow(name="process-subflow", log_prints=True)
def process_subflow(ids: list[int], config: PipelineConfig) -> list[str]:
    return process_ids(ids, config)


@flow(name="param-passing-demo", log_prints=True)
def param_passing_pipeline(env: str = "dev", batch_size: int = 5, dry_run: bool = True):
    # Build shared config in the parent and pass it to both subflows
    cfg = PipelineConfig(env=env, batch_size=batch_size, dry_run=dry_run)
    print(f"Config: {cfg}")

    ids      = fetch_subflow(config=cfg, offset=100)
    records  = process_subflow(ids=ids, config=cfg)   # return value of one subflow → input of next

    print(f"Pipeline complete: {len(records)} records processed")
    return records


param_passing_pipeline(env="staging", batch_size=4, dry_run=True)

### What just happened?

- A **`PipelineConfig` dataclass** was built once in the parent and passed to both subflows — a clean alternative to environment variables or global state.
- The **return value of `fetch_subflow`** (`ids`) was fed directly as an argument to `process_subflow` — this is the standard chain pattern.
- **`dry_run=True`** is respected inside the subflow's task without the parent having to know about it.
- This pattern scales: a config object is easy to extend without changing every function signature.

## Step 4 · How subflow states bubble up to the parent

By default, if a subflow run reaches a **`Failed`** or **`Crashed`** terminal state, Prefect raises a `prefect.exceptions.CrashException` or re-raises the underlying exception in the parent flow, causing the parent to also fail.

State propagation rules:

| Subflow terminal state | Parent behaviour (default) |
|---|---|
| `Completed` | Parent continues normally |
| `Failed` | Exception raised at the call site in the parent |
| `Crashed` | Exception raised at the call site in the parent |
| `Cancelled` | Exception raised at the call site in the parent |

You can prevent propagation by calling the subflow with **`return_state=True`** and handling the state yourself (Step 5).

In [ ]:
from prefect import flow, task


@task
def validate_schema(rows: list[dict]) -> list[dict]:
    """Raises if required keys are missing."""
    required = {"id", "amount"}
    for r in rows:
        missing = required - r.keys()
        if missing:
            raise ValueError(f"Row missing keys: {missing} — row={r}")
    return rows


@flow(name="validate-subflow", log_prints=True)
def validate_subflow(rows: list[dict]) -> list[dict]:
    """Subflow that fails if schema validation fails."""
    valid = validate_schema(rows)
    print(f"  [validate] all {len(valid)} rows pass schema check")
    return valid


@flow(name="state-propagation-demo", log_prints=True)
def state_propagation_demo(bad_data: bool = False):
    rows_ok  = [{"id": 1, "amount": 10.0}]
    rows_bad = [{"id": 2}]                   # missing 'amount'
    rows     = rows_bad if bad_data else rows_ok

    # This call will raise in the parent if validate_subflow fails
    validated = validate_subflow(rows)
    print(f"  Validated rows: {validated}")
    return validated


# ── Run with good data — parent completes ─────────────────────────────────────
print("=== Good data ===")
state_propagation_demo(bad_data=False)

# ── Run with bad data — subflow fails, parent fails ───────────────────────────
print("\n=== Bad data (subflow failure bubbles up) ===")
try:
    state_propagation_demo(bad_data=True)
except Exception as exc:
    print(f"  Parent caught exception: {type(exc).__name__}: {exc}")

### What just happened?

- When `validate_subflow` failed (bad data), the exception **propagated to the parent** and `state_propagation_demo` also failed.
- The parent's exception is the same `ValueError` that the task inside the subflow raised — the subflow is transparent to exceptions.
- **This is intentional**: a failed stage should stop the pipeline by default, not silently pass garbage to the next stage.
- Use `return_state=True` (next step) when you want to handle failure gracefully instead.

## Step 5 · `return_state=True` — inspect a subflow's final state

Passing `return_state=True` to a subflow call makes Prefect return the **`State` object** instead of raising on failure. This lets the parent make branching decisions:

```python
state = my_subflow(args, return_state=True)
if state.is_failed():
    # handle gracefully
```

Useful `State` methods:

| Method | Returns |
|---|---|
| `state.is_completed()` | `True` if terminal Completed |
| `state.is_failed()` | `True` if terminal Failed |
| `state.name` | String name, e.g. `"Completed"`, `"Failed"` |
| `state.result(raise_on_failure=False)` | The return value, or the exception |
| `state.timestamp` | When the state was entered |

In [ ]:
from prefect import flow, task


@task
def risky_transform(rows: list[dict], strict: bool) -> list[dict]:
    """Fails in strict mode if any row has a negative amount."""
    negatives = [r for r in rows if r.get("amount", 0) < 0]
    if strict and negatives:
        raise ValueError(f"Found {len(negatives)} rows with negative amounts")
    return [r for r in rows if r.get("amount", 0) >= 0]  # drop negatives in lenient mode


@flow(name="transform-strict", log_prints=True)
def transform_strict_subflow(rows: list[dict], strict: bool = True) -> list[dict]:
    return risky_transform(rows, strict)


@flow(name="resilient-parent", log_prints=True)
def resilient_parent_flow():
    dirty_rows = [
        {"id": 1, "amount": 15.0},
        {"id": 2, "amount": -3.0},   # negative — will trigger strict failure
        {"id": 3, "amount": 8.0},
    ]

    # ── Try strict mode first ─────────────────────────────────────────────────
    strict_state = transform_strict_subflow(
        rows=dirty_rows, strict=True,
        return_state=True              # <── get State object, don't raise
    )

    print(f"  Strict subflow state : {strict_state.name}")
    print(f"  is_failed()          : {strict_state.is_failed()}")

    if strict_state.is_failed():
        exc = strict_state.result(raise_on_failure=False)
        print(f"  Failure reason       : {exc}")
        print("  Falling back to lenient mode...")

        # ── Fallback: lenient mode ────────────────────────────────────────────
        lenient_state = transform_strict_subflow(
            rows=dirty_rows, strict=False,
            return_state=True
        )
        print(f"  Lenient subflow state: {lenient_state.name}")
        cleaned = lenient_state.result(raise_on_failure=True)
    else:
        cleaned = strict_state.result(raise_on_failure=True)

    print(f"  Final cleaned rows: {cleaned}")
    return cleaned


resilient_parent_flow()

### What just happened?

- `return_state=True` gave the parent a **`State` object** instead of raising an exception when the strict subflow failed.
- `state.is_failed()` enabled the parent to detect failure and **branch to a fallback strategy**.
- `state.result(raise_on_failure=False)` safely extracted the underlying exception for logging.
- **This pattern is ideal for optional enrichment subflows**: try the expensive path, fall back to a cheaper one if it fails.

In [ ]:
# ─── Challenge ────────────────────────────────────────────────────────────────
# Challenge: Build a three-stage ETL parent flow with subflows for each stage.
#
# Requirements:
#   1. ingest_subflow(source: str) → list[dict]
#        Read data from `source` (simulate: return a hard-coded list of 5 dicts
#        with keys "id" and "score").
#
#   2. score_subflow(rows: list[dict], threshold: float) → list[dict]
#        Keep only rows where score >= threshold.
#        Raise ValueError if NO rows survive the filter.
#
#   3. report_subflow(rows: list[dict], label: str) → str
#        Return a summary string: "label: N rows, avg score = X.XX".
#
#   4. Parent flow: call score_subflow with return_state=True.
#        If it fails (all rows filtered), fall back to threshold=0.0
#        and log a warning before calling report_subflow.
#
# Test by running the parent with threshold=0.9 (should trigger fallback)
# and threshold=0.5 (should succeed normally).
# ──────────────────────────────────────────────────────────────────────────────

from prefect import flow, task


# TODO: implement ingest_subflow


# TODO: implement score_subflow (raise if no rows survive)


# TODO: implement report_subflow


# TODO: implement parent flow with return_state=True fallback logic


# Uncomment to test:
# main_flow(source="experiment_results", threshold=0.9)  # triggers fallback
# main_flow(source="experiment_results", threshold=0.5)  # normal path

---
## Day 5 key concepts recap

| Concept | What to remember |
|---|---|
| `@flow` inside `@flow` | Any flow called from a flow becomes a subflow — tracked as a nested run |
| Parameters | Passed as ordinary function arguments; no special API |
| Return values | A subflow's return value is available to the parent like any function |
| State propagation | Failed subflow raises in parent by default — pipeline stops |
| `return_state=True` | Returns `State` object; lets parent branch on success/failure |
| `state.result(raise_on_failure=False)` | Extract the return value or exception without re-raising |
| Domain decomposition | Split by Ingest / Transform / Load for independent scheduling and retry |

> **Tip:** Break large pipelines into subflows by domain (ingest, transform, load) — each subflow can be scheduled, tested, and retried independently.

---
## What's next
**Day 6** → Results, Artifacts, and State Persistence — persist task results to disk with `LocalFileSystemResultStorage` and publish human-readable summaries as Prefect artifacts.

Mark Day 5 complete in your [tracker](../index.html).